In [7]:
import pandas as pd
import sqlite3 as sq
import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import fiona
import os
import io
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.functions import read_census_table, read_wp23_table

Read in base year from the Woods and Poole 2023 Vintage Database. The dependencies for these have been run, and the data is written out in the file "Demographic Model Base Year Inputs.ipynb, found in the repository Woods-and-Poole-2023.

In [9]:
dembase = read_wp23_table('DemographicBaseYear')

In [11]:
dembase.head(2)

,NAME,Average Household Size,Population,GQ Population,Housing Units,HH Pop,Last Update Pushed_20260120,Households
0,"Cannon County, Tennessee",2.48,15063,108,6518,14955,None,6030
1,"Cheatham County, Tennessee",2.60,42254,330,17801,41924,None,16125


In [12]:
thelist = ['Trigg County, Kentucky', 'Christian County, Kentucky']
dembase_cos = dembase.loc[dembase['NAME'].isin(thelist)]

In [13]:
dembase_cos

,NAME,Average Household Size,Population,GQ Population,Housing Units,HH Pop,Last Update Pushed_20260120,Households
17,"Christian County, Kentucky",2.62,72032,4918,29405,67114,None,25616
19,"Trigg County, Kentucky",2.42,14369,58,7697,14311,None,5914


In [53]:
file = 'https://www2.census.gov/geo/tiger/TIGER2020/TABBLOCK20/tl_2020_21_tabblock20.zip'
geo = gpd.read_file(file)

In [54]:
geo.head(1)

,STATEFP20,COUNTYFP20,TRACTCE20,BLOCKCE20,GEOID20,NAME20,MTFCC20,UR20,UACE20,UATYPE20,FUNCSTAT20,ALAND20,AWATER20,INTPTLAT20,INTPTLON20,HOUSING20,POP20,geometry
0,21,017,030600,1002,210170306001002,Block 1002,G5040,R,NaN,NaN,S,934479,0,+38.2163349,-084.0124264,6,10,"POLYGON ((-84.02373 38.21494, -84.02316 38.215..."


In [55]:
geo['GEO_ID'] = geo['GEOID20'].astype(str)
geo_join = geo[['GEO_ID', 'geometry']]

In [56]:
geo_join.head(1)

,GEO_ID,geometry
0,210170306001002,"POLYGON ((-84.02373 38.21494, -84.02316 38.215..."


In [57]:
geo_join.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 132662 entries, 0 to 132661
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype   
---  ------    --------------   -----   
 0   GEO_ID    132662 non-null  str     
 1   geometry  132662 non-null  geometry
dtypes: geometry(1), str(1)
memory usage: 2.0 MB


In [62]:
dec = read_census_table('PL_2020_BLKKY')
counties = ['Trigg County', 'Christian County']
dec = dec.loc[dec['County'].isin(counties)]
dec = dec[['GEO_ID', 'Household Population', 'County']]

In [63]:
dec.head(1)

,GEO_ID,Household Population,County
21818,1000000US210472001002003,17,Christian County


In [68]:
dec['GEO_ID'] = dec['GEO_ID'].str[9:].astype(str)

In [69]:
dec.info()

<class 'pandas.DataFrame'>
Index: 3001 entries, 21818 to 123693
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   GEO_ID                3001 non-null   str  
 1   Household Population  3001 non-null   int64
 2   County                3001 non-null   str  
dtypes: int64(1), str(2)
memory usage: 93.8 KB


In [70]:
dec.head(2)

,GEO_ID,Household Population,County
21818,210472001002003,17,Christian County
21819,210472001002006,0,Christian County


In [71]:
dec_geo = dec.merge(geo_join, how = "left", on = "GEO_ID")

In [72]:
dec_geo.tail(2)

,GEO_ID,Household Population,County,geometry
2999,212219802001019,0,Trigg County,"POLYGON ((-87.78198 36.63588, -87.78171 36.637..."
3000,212219802001020,0,Trigg County,"POLYGON ((-87.76888 36.63583, -87.76842 36.636..."


In [75]:
trigg = dec_geo.loc[dec_geo['County'] == 'Trigg County'].reset_index(drop = True)
christian = dec_geo.loc[dec_geo['County'] == 'Christian County'].reset_index(drop = True)

In [76]:
trigg.head()

,GEO_ID,Household Population,County,geometry
0,212219701003003,4,Trigg County,"POLYGON ((-87.87697 36.96011, -87.87566 36.959..."
1,212219701003006,16,Trigg County,"POLYGON ((-87.87628 36.95714, -87.87603 36.957..."
2,212219701003009,12,Trigg County,"POLYGON ((-87.93644 36.92322, -87.93629 36.923..."
3,212219701003013,11,Trigg County,"POLYGON ((-87.8564 36.92651, -87.85621 36.9266..."
4,212219701003016,31,Trigg County,"POLYGON ((-87.84762 36.92303, -87.84755 36.923..."


In [104]:
trigg["Allocation"] = (trigg["Household Population"] / trigg["Household Population"].sum())*100
christian["Allocation"] = (christian["Household Population"] / christian["Household Population"].sum())*100

trigg_control = dembase.loc[dembase["NAME"] == "Trigg County, Kentucky","HH Pop"].iloc[0]
christian_control = dembase.loc[dembase["NAME"] == "Christian County, Kentucky","HH Pop"].iloc[0]

trigg["HH Pop Alloc"] = (trigg["Allocation"])/100*trigg_control
christian["HH Pop Alloc"] = (christian["Allocation"])/100*christian_control

trigg_diff = abs((trigg['HH Pop Alloc'] - trigg['Household Population'])/trigg['Household Population'])*100
christian_diff = abs((christian['HH Pop Alloc'] - christian['Household Population'])/christian['Household Population'])*100

In [106]:
christian.head()

,GEO_ID,Household Population,County,geometry,Allocation,HH Pop Alloc,Diff
0,210472001002003,17,Christian County,"POLYGON ((-87.48637 36.8692, -87.48623 36.8694...",0.025039,16.804448,1.150306
1,210472001002006,0,Christian County,"POLYGON ((-87.48955 36.86747, -87.48942 36.867...",0.000000,0.000000,NaN
2,210472001002009,30,Christian County,"POLYGON ((-87.48752 36.86756, -87.48729 36.867...",0.044186,29.654908,1.150306
3,210472001002013,0,Christian County,"POLYGON ((-87.48913 36.86704, -87.4889 36.8673...",0.000000,0.000000,NaN
4,210472001002016,0,Christian County,"POLYGON ((-87.50234 36.86766, -87.50227 36.867...",0.000000,0.000000,NaN


In [115]:
county_pop = read_census_table('PEP_2023V')

In [116]:
counties = ['Stewart County, Tennessee', 'Montgomery County, Tennessee', 'Christian County, Kentucky', 'Trigg County, Kentucky']
county_pop = county_pop.loc[(county_pop['NAME'].isin(thelist)) & (county_pop['Year'] == '2023')]

In [117]:
county_pop.head()

,NAME,Year,Population,GQ Population,Housing Units,HH Pop,Last Update Pushed_20260120
10629,"Christian County, Kentucky",2023,72032,4918,29405,67114,None
10716,"Trigg County, Kentucky",2023,14369,58,7697,14311,None


In [122]:
county_dec = read_census_table('PL_2020')

In [133]:
county_dec = read_census_table('PL_2020')
counties = ['Stewart County, Tennessee', 'Montgomery County, Tennessee', 'Christian County, Kentucky', 'Trigg County, Kentucky']
county_dec = county_dec.loc[(county_dec['NAME'].isin(thelist))]

In [135]:
county_dec

,index,NAME,GEO_ID,Population,White Alone,White Alone %,Non-White Population: Race,Black or African American Alone,Black or African American Alone %,Black or African American Alone % Non-White Population,...,Group Quarters % Noninstitutionalized:College or University Student Housing,Group Quarters %:Noninstitutionalized in College or University Student Housing,"Group Quarters:Noninstitutionalized, Military Quarters",Group Quarters % Noninstitutionalized:Military Quarters,Group Quarters %:Noninstitutionalized in Military Quarters,Group Quarters%:Total Population in Military Quarters,"Group Quarters:Noninstitutionalized, Other",Group Quarters % Noninstitutionalized:Other,Group Quarters %:Noninstitutionalized in Other,Last Update Pushed_20261006
118,118,"Christian County, Kentucky",0500000US21047,72748.0,48279.0,66.364711,24469.0,15527.0,21.343542,63.455801,...,0.0,0.0,3728.0,98.990972,76.818463,5.12454,3728.0,98.990972,76.818463,None
205,205,"Trigg County, Kentucky",0500000US21221,14061.0,12309.0,87.540004,1752.0,952.0,6.770500,54.337900,...,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.0,0.000000,0.000000,None
